# 🎬 GaussianEditor C4D — Training 3DGS depuis renders Cinema 4D

**Avant de commencer :**
1. `Runtime` → `Change runtime type` → **T4 GPU** (gratuit)
2. Assure-toi que ton dossier est dans Google Drive :
   ```
   Mon Drive/
     gs_renders/
       mon_objet/
         images/
           0000.png
           0001.png
           ...
           0035.png
         transforms.json
   ```

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 1 — Vérifier le GPU
# ═══════════════════════════════════════════
!nvidia-smi
print('\n✅ GPU disponible — on peut commencer')

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 2 — Connecter Google Drive
# ═══════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive connecté')

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 3 — Paramètres (modifier ici)
# ═══════════════════════════════════════════

# Nom du dossier dans Drive > gs_renders >
OBJECT_NAME = 'mon_objet'   # <-- changer ici

# Chemins
import os
DATA_DIR   = f'/content/drive/MyDrive/gs_renders/{OBJECT_NAME}'
OUTPUT_DIR = f'/content/drive/MyDrive/gs_output/{OBJECT_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Vérifier que les fichiers existent
images = os.listdir(os.path.join(DATA_DIR, 'images'))
print(f'📁 Dossier     : {DATA_DIR}')
print(f'🖼️  Images       : {len(images)} fichiers')
print(f'📄 transforms  : {os.path.exists(os.path.join(DATA_DIR, "transforms.json"))}')
print(f'💾 Sortie      : {OUTPUT_DIR}')

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 4 — Installer Nerfstudio
# (attendre ~5 min la première fois)
# ═══════════════════════════════════════════
print('Installation de nerfstudio...')
!pip install nerfstudio -q
print('\n✅ Nerfstudio installé')

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 5 — Lancer le training
# (~15-30 min selon la complexité de l'objet)
# ═══════════════════════════════════════════
import subprocess, sys

cmd = [
    'ns-train', 'splatfacto',
    '--data', DATA_DIR,
    '--output-dir', OUTPUT_DIR,
    '--max-num-iterations', '30000',
    '--pipeline.model.sh-degree', '3',
    '--vis', 'wandb',    # désactiver si pas de compte wandb
]

# Sans wandb (plus simple)
cmd = [
    'ns-train', 'splatfacto',
    '--data', DATA_DIR,
    '--output-dir', OUTPUT_DIR,
    '--max-num-iterations', '30000',
    '--pipeline.model.sh-degree', '3',
    '--vis', 'viewer',
    '--viewer.quit-on-train-completion', 'True',
]

print('🚀 Training en cours...')
print(' '.join(cmd))
!ns-train splatfacto \
    --data {DATA_DIR} \
    --output-dir {OUTPUT_DIR} \
    --max-num-iterations 30000 \
    --pipeline.model.sh-degree 3 \
    --vis none

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 6 — Trouver le config.yml généré
# ═══════════════════════════════════════════
import glob

configs = glob.glob(f'{OUTPUT_DIR}/**/config.yml', recursive=True)
if not configs:
    print('❌ Aucun config.yml trouvé — le training a peut-être échoué')
else:
    CONFIG_PATH = sorted(configs)[-1]  # le plus récent
    print(f'✅ Config trouvé : {CONFIG_PATH}')

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 7 — Exporter le .ply Gaussian
# ═══════════════════════════════════════════
PLY_OUTPUT = os.path.join(OUTPUT_DIR, 'gaussian_splat')
os.makedirs(PLY_OUTPUT, exist_ok=True)

!ns-export gaussian-splat \
    --load-config {CONFIG_PATH} \
    --output-dir {PLY_OUTPUT}

# Vérifier le fichier
plys = glob.glob(f'{PLY_OUTPUT}/**/*.ply', recursive=True)
if plys:
    ply_path = plys[0]
    size_mb = os.path.getsize(ply_path) / (1024*1024)
    print(f'\n✅ PLY généré : {ply_path}')
    print(f'   Taille     : {size_mb:.1f} MB')
    print(f'\n📥 Télécharge ce fichier depuis Google Drive :')
    print(f'   Drive > gs_output > {OBJECT_NAME} > gaussian_splat > *.ply')
else:
    print('❌ Aucun .ply trouvé')

In [ ]:
# ═══════════════════════════════════════════
# CELLULE 8 — Commandes WSL2 pour l'injection
# ═══════════════════════════════════════════
print('Après avoir téléchargé le .ply sur ton PC Windows :')
print()
print('Dans WSL2 :')
print()
print('# 1. Charger ta scène')
print('curl -X POST http://127.0.0.1:8086/load \\')
print('     -H "Content-Type: application/json" \\')
print('     -d \'{"ply_path":"/mnt/c/Users/MSI/.../scene/chair.ply"}\'')
print()
print('# 2. Ajouter ton objet généré')
print('curl -X POST http://127.0.0.1:8086/add_ply \\')
print('     -H "Content-Type: application/json" \\')
print('     -d \'{"ply_path":"/mnt/c/Users/MSI/Downloads/mon_objet.ply","source":"3dgs"}\'')